In [8]:
import pandas as pd 
import os 
import json 
import numpy as np
from os.path import dirname

pd.set_option("display.max_columns", None)
root_path = dirname(os.getcwd())
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/comuzzi/_processed"
data_dir_graphs = root_path + "/data/datasets/comuzzi/graphs_repair/"
data_dir_ablation = root_path + "/data/datasets/ablation/" 
os.makedirs(data_dir_ablation, exist_ok=True)
print(root_path, data_dir_processed, data_dir_graphs, data_dir_ablation, sep="\n")

/home/danbi/Projects/SANAGRAPH
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/_processed
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/
/home/danbi/Projects/SANAGRAPH/data/datasets/ablation/


In [3]:
from pipeline import (
    build_split_graphs,
    build_test_graphs_for_type,
    get_feature_variants,
    convert_feature_name,
)

In [9]:
with open("dataset_features.json", 'r') as file:
    dataset_info = json.load(file)

In [10]:
list(dataset_info.keys())

['BPI_Challenge_2013_open_problems',
 'sp2020',
 'Helpdesk',
 'BPI20_RequestForPayment',
 'BPI Challenge 2017 - Offer log',
 'BPI_Challenge_2012_W_Complete',
 'BPI_Challenge_2012_A',
 'bpi_2012_CZ',
 'bpi_2013_CZ',
 'large_log_CZ',
 'small_log_CZ',
 'sp2020_CZ',
 'BPI20_RequestForPayment_CZ']

In [11]:
dataset = "BPI20_RequestForPayment_CZ"

In [12]:
ACT_TIME_ONLY = False

In [13]:
dataset_info = dataset_info[dataset]

In [14]:
variants = get_feature_variants(dataset_info)
print(f"{len(variants)} Variants to generate for {dataset}:")
for excluded_feature, cat, num in variants:
    print(f"  - ablate_{convert_feature_name(excluded_feature)} (excluded feature: {excluded_feature})")

10 Variants to generate for BPI20_RequestForPayment_CZ:
  - ablate_org_resource (excluded feature: org:resource)
  - ablate_Activity (excluded feature: Activity)
  - ablate_org_role (excluded feature: org:role)
  - ablate_case_Project (excluded feature: case:Project)
  - ablate_case_Task (excluded feature: case:Task)
  - ablate_case_OrganizationalEntity (excluded feature: case:OrganizationalEntity)
  - ablate_case_Activity (excluded feature: case:Activity)
  - ablate_case_RfpNumber (excluded feature: case:RfpNumber)
  - ablate_time_timestamp (excluded feature: time:timestamp)
  - ablate_case_RequestedAmount (excluded feature: case:RequestedAmount)


In [15]:
nan_methods = ["odd", "even", "random", "window", "attr_level"]
masked_datasets = {key: pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_masked_{key}_all.csv") for key in nan_methods}

tab_all = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_all.csv")
tab_train = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_test.csv")

In [16]:
all_categorical = dataset_info["categorical"]

for k in all_categorical:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")
    for k_m in masked_datasets:
        masked_datasets[k_m][k] = masked_datasets[k_m][k].astype("object")

tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)
for k_m in masked_datasets:
    masked_datasets[k_m]["CaseID"] = masked_datasets[k_m]["CaseID"].astype(np.str_)

In [18]:
import pickle

In [ ]:
for excluded_feature, categorical_columns, real_value_columns in variants:

    safe_feature = convert_feature_name(excluded_feature)
    variant_dir = data_dir_ablation + f"{dataset}/ablate_{safe_feature}/"
    os.makedirs(variant_dir, exist_ok=True)

    print(f"\n=== Variant: ablate_{safe_feature} ===")

    for split_name, tab_split in [("TRAIN", tab_train), ("VALID", tab_valid), ("TEST", tab_test)]:
        print(f"  {split_name}...")
        X = build_split_graphs(tab_all, tab_split, categorical_columns, real_value_columns, masked_datasets, nan_methods)
        with open(variant_dir + f"{split_name}_V2_repair.pkl", "wb") as f:
            pickle.dump(X, f)
        del X

    for test_type in nan_methods:
        print(f"  TEST ({test_type})...")
        X = build_test_graphs_for_type(tab_all, tab_test, categorical_columns, real_value_columns, masked_datasets, nan_methods, mask_type=test_type)
        with open(variant_dir + f"TEST_V2_repair_{test_type}.pkl", "wb") as f:
            pickle.dump(X, f)
        del X

print("\nTutte le varianti generate.")


=== Variant: ablate_org_resource ===
  TRAIN...


  0%|          | 0/4131 [00:00<?, ?it/s]